# Long Short-Term Memory (LSTM) - TensorFlow / Keras

**Goal:** Classify Reuters newswire topics.

This notebook favors clear, production-style structure: seeded runs,
explicit data preparation, small reusable modules, and compact
training loops that can be expanded for larger experiments.


## Architecture Notes

- **What it learns:** Gates decide what to write, keep, forget, and expose from memory.
- **Where it is used:** longer text and time-series tasks where memory retention matters.
- **Why it works:** the architecture builds a useful bias into the computation, so the model does not need to rediscover that structure from data alone.
- **Output to expect:** classification models return class scores/probabilities, reconstruction models return reconstructed inputs, and generative models return new or denoised samples.


## Visual Intuition

Run this cell before or after training. It is lightweight and framework-independent, so it explains the network idea without requiring a long training run.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
fig.suptitle("Long Short-Term Memory: intuition, signal flow, and output", fontsize=13)
axes[0].axis("off")
layers = ['input', 'gates', 'memory']
xs = np.linspace(0.1, 0.9, len(layers))
for xpos, label in zip(xs, layers):
    axes[0].scatter([xpos], [0.55], s=1200, color="#4C78A8", alpha=0.18, edgecolors="#4C78A8")
    axes[0].text(xpos, 0.55, label, ha="center", va="center", fontsize=9)
for a, b in zip(xs[:-1], xs[1:]):
    axes[0].annotate("", xy=(b - 0.045, 0.55), xytext=(a + 0.045, 0.55), arrowprops=dict(arrowstyle="->", lw=1.5))
axes[0].set_title("How data moves")
x = np.linspace(-3, 3, 160)
y = 1/(1+np.exp(-x))
axes[1].plot(x, y, color="#F58518", lw=2)
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_title("Toy behavior")
axes[1].grid(alpha=0.25)
values = np.array([.15,.25,.60])
axes[2].bar(range(len(values)), values, color=["#54A24B", "#E45756", "#72B7B2", "#B279A2"][:len(values)])
axes[2].set_title("Typical output")
axes[2].set_xticks(range(len(values)))
axes[2].set_xticklabels(['forget', 'input', 'output'])
axes[2].grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(SEED)
print(f"TensorFlow: {tf.__version__}")
samples, seq_len, features = 1400, 30, 3
X = np.random.normal(size=(samples, seq_len, features)).astype("float32")
signal = X[:, -10:, 0].mean(axis=1) + 0.5 * X[:, :10, 1].mean(axis=1)
y = (signal > 0.05).astype("int64")
X_train, X_test = X[:1000], X[1000:]
y_train, y_test = y[:1000], y[1000:]


In [ ]:
model = keras.Sequential([
    layers.Input(shape=(30, 3)),
    layers.LSTM(32),
    layers.Dropout(0.2),
    layers.Dense(2, activation="softmax"),
])
model.compile(optimizer=keras.optimizers.AdamW(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()


In [ ]:
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=64)
